# 🧬 RS-SMC: Randomized Smoothing Sequential Monte Carlo (Table 8 Replication & Scaling)

> **Mục tiêu**: Tái lập và nâng cấp toàn diện thí nghiệm **Bảng 8 (Table 8)** trong bài báo gốc ICML 2026 (*"Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models"*), khảo sát sự tương tác đa hạt mục tiêu của **Sequential Monte Carlo (SMC)**, **Randomized Smoothing SMC (RS-SMC)**, **Best-of-N (BoN)**, và sự kết hợp trực giao với **LiDAR** trên **Stable Diffusion v1.5** và **Stable Diffusion XL (SDXL)**.

---

### 💡 Trực Giác Toán Học: Tại Sao Làm Mịn Reward Trên Ước Lượng Tweedie $\hat{x}_0$?
1. **Độ lệch Tweedie (Tweedie Approximation Drift)**: Ở các bước khử nhiễu trung gian $t \in [20, 80]$, ước lượng Tweedie $\hat{x}_0(x_t) = \mathbb{E}[x_0 \mid x_t]$ là một xấp xỉ mờ, thiếu chi tiết vi mô và có độ lệch phân phối so với ảnh hội tụ cuối cùng $x_0^*$.
2. **Hàm Reward Thô $R_{\text{raw}}$ Rất Nhạy Cảm**: Các mô hình như ImageReward có độ dốc Lipschitz $L_{\text{raw}}$ cực kỳ lớn và nhiều gai nhọn đối kháng. Một sai lệch nhỏ trong ảnh Tweedie có thể làm điểm reward bị **nhảy vọt hoặc rơi tự do (wild fluctuation)**, gây:
   - *Premature Pruning*: Loại bỏ nhầm hạt xuất sắc chỉ vì ảnh mờ Tweedie tạm rơi vào vùng trũng.
   - *Spurious Collapse*: Nhân bản hạt lỗi ra toàn bộ quần thể ($ESS \to 1.0$) vì một đốm nhiễu may rủi kích hoạt điểm cao giả tạo.
3. **Randomized Smoothing Giải Quyết Triệt Để**: Tích chập Gauss $\hat{R}_{\sigma}(x) = \mathbb{E}_{\epsilon}[R(x+\epsilon)]$ làm phẳng toàn bộ các gai nhọn, giới hạn độ dốc Lipschitz toàn cục $\|\nabla \hat{R}_{\sigma}\| \le \frac{\sqrt{2/\pi}}{\sigma} \|R\|_{\infty}$. Nhờ đó:
   $$|\hat{R}_{\sigma}(\hat{x}_0) - \hat{R}_{\sigma}(x_0^*)| \le L_{\sigma} \|\hat{x}_0 - x_0^*\|$$
   Reward của ước lượng Tweedie không còn bị giật cục, luôn bám sát điểm số thực tế của quỹ đạo đích, giúp SMC tái chọn mẫu chính xác vào các vùng nghiệm chất lượng cao.


## 1. 🖥️ Kiểm Tra Phần Cứng GPU (Khuyến nghị NVIDIA T4 / V100 / A100)


In [ ]:
!nvidia-smi

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\n✅ GPU khả dụng: {gpu_name} ({total_vram:.2f} GiB VRAM)")
    if "A100" in gpu_name:
        print("🌟 Bạn đang chạy trên NVIDIA A100 chuẩn benchmark bài báo ICML 2026!")
    else:
        print("ℹ️ GPU phù hợp cho thực nghiệm. SD 1.5 chạy tốt trên mọi GPU >= 8 GiB; SDXL khuyến nghị GPU >= 16 GiB.")
else:
    print("❌ Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")


## 2. 📁 Gắn Kết Google Drive Cá Nhân (Tự Động Lưu Kết Quả Chống Mất Dữ Liệu)


In [ ]:
# @title 📁 Gắn kết Google Drive & Thiết lập thư mục lưu trữ
import os
from google.colab import drive

MOUNT_DRIVE = True # @param {type:"boolean"}
DRIVE_SMC_DIR = None

if MOUNT_DRIVE:
    try:
        drive.mount('/content/drive')
        base_drive = None
        for cand in ['/content/drive/MyDrive', '/content/drive/My Drive', '/content/drive']:
            if os.path.exists(cand):
                base_drive = cand
                break
        if base_drive is None:
            base_drive = '/content/drive/MyDrive'

        DRIVE_SMC_DIR = os.path.join(base_drive, "RS-LiDAR", "results", "table8_smc")
        os.makedirs(DRIVE_SMC_DIR, exist_ok=True)
        print(f"\n✅ Đã gắn kết Google Drive thành công!")
        print(f"📁 Thư mục tự động sao lưu trên Drive: {DRIVE_SMC_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể mount Google Drive: {e}. Kết quả sẽ chỉ được lưu tạm thời trên đĩa Colab.")
        DRIVE_SMC_DIR = None
else:
    print("ℹ️ Bỏ qua gắn kết Google Drive. Kết quả sẽ lưu cục bộ tại 'Target_samples/'.")


## 3. 📂 Đồng Bộ Mã Nguồn Từ GitHub & Định Tuyến Thư Mục


In [ ]:
# @title 📥 Bước 3: Clone / Update Repository RS-LiDAR và Chuyển Thư Mục
import os

!git clone https://github.com/leekwanreal/RS-LiDAR.git /content/RS-LiDAR || git -C /content/RS-LiDAR pull origin main
%cd /content/RS-LiDAR

assert os.path.exists("LiDAR_sampling.py"), f"❌ Chưa tìm thấy LiDAR_sampling.py trong {os.getcwd()}!"
print(f"\n✅ Đã ở đúng thư mục làm việc: {os.getcwd()}")


## 4. 📦 Cài Đặt Thư Viện Cần Thiết (Triệt Tiêu Lỗi Protobuf)


In [ ]:
# 1. Tắt backend TensorFlow để loại bỏ hoàn toàn xung đột Protobuf runtime_version
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

# 2. Cài đặt các thư viện tương thích PyTorch 2.x chuẩn Table 2 Replication
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q matplotlib tqdm scipy seaborn pandas tabulate

print('\n✅ Cài đặt môi trường chuẩn xác 100% (Không xung đột Protobuf)!')


## 5. 🎛️ Cấu Hình Thực Nghiệm (Interactive Form)


In [ ]:
# @title ⚙️ Chọn Siêu Tham Số Thực Nghiệm SMC & RS-SMC

# @markdown **1. Lựa chọn Mô hình (Model Architecture)**:
MODEL_CHOICE = "sdv1.5" # @param ["sdv1.5", "sdxl"]

# @markdown **2. Phương pháp lấy mẫu (Sampling Method)**:
METHOD = "rs_smc" # @param ["rs_smc", "vanilla_smc", "bon", "rs_smc_lidar", "smc_lidar"]

# @markdown **3. Số hạt mục tiêu N (Target Particles)**:
NUM_PARTICLES = 4 # @param [1, 2, 4, 8, 16, 32, 64]
# @markdown *(Table 8 khảo sát dải hạt: N ∈ {1, 2, 4, 8, 16, 32, 64})*

# @markdown **4. Độ lệch chuẩn nhiễu làm mịn (SMC Sigma)**:
SMC_SIGMA = 0.25 # @param {type:"number"}
# @markdown *(Gợi ý: 0.25 cho SD 1.5; 0.10 cho SDXL; Tự động về 0.0 nếu chọn Vanilla SMC)*

# @markdown **5. Số mẫu Monte Carlo (SMC M)**:
SMC_NUM_MC_SAMPLES = 4 # @param {type:"integer"}
# @markdown *(Gợi ý: 4 mẫu Monte Carlo là điểm cân bằng lý tưởng giữa hiệu năng và thời gian)*

# @markdown **6. Kỹ thuật Common Random Numbers (CRN)**:
SMC_USE_CRN = False # @param {type:"boolean"}
# @markdown *(Nếu True: Dùng chung 1 vector nhiễu cho toàn bộ N hạt trong prompt để triệt tiêu phương sai giữa các hạt)*

# @markdown **7. Số bước khử nhiễu (Denoising Steps)**:
NUM_INFERENCE_STEPS = 100 # @param {type:"integer"}

# @markdown **8. Tần suất Resample (Resampling Frequency)**:
RESAMPLE_FREQUENCY = 20 # @param {type:"integer"}
# @markdown *(Chuẩn bài báo Table 8: Resample mỗi 20 bước khử nhiễu)*

# @markdown **9. Nhiệt độ SMC (FK Lambda)**:
FK_LMBDA = 10.0 # @param {type:"number"}

# @markdown **10. Cơ chế chọn prompt (Prompt Selection Strategy)**:
PROMPT_SELECTION_MODE = "stratified_by_category" # @param ["stratified_by_category", "first_n", "custom_indices"]

# @markdown **11. Số prompt cho mỗi category (nếu chọn stratified)**:
PROMPTS_PER_CATEGORY = 2 # @param {type:"integer"}
# @markdown *(2 prompts x 6 categories = 12 prompts GenEval chuẩn)*

# @markdown **12. Số lượng prompts (nếu chọn first_n)**:
NUM_PROMPTS = 4 # @param {type:"integer"}

# @markdown **13. Danh sách prompt indices cụ thể (nếu chọn custom_indices)**:
CUSTOM_PROMPT_INDICES = "80,98,95,93" # @param {type:"string"}

# @markdown **14. Tên thử nghiệm tùy chọn (Run Name - để trống để đặt tên tự động theo tham số)**:
RUN_NAME = "" # @param {type:"string"}

# @markdown **15. Tự động sao lưu kết quả sang Google Drive**:
SAVE_TO_DRIVE = True # @param {type:"boolean"}

# Auto-Dispatch siêu tham số theo kiến trúc mô hình
is_sdxl = (MODEL_CHOICE == "sdxl")
model_name = "stabilityai/stable-diffusion-xl-base-1.0" if is_sdxl else "runwayml/stable-diffusion-v1-5"
guidance_scale = 8.0 if is_sdxl else 12.5
effective_sigma = 0.10 if (is_sdxl and "rs_" in METHOD) else (SMC_SIGMA if "rs_" in METHOD else 0.0)
effective_m = SMC_NUM_MC_SAMPLES if "rs_" in METHOD else 1
lookahead_dataset = "sdxl_dmd1_100" if is_sdxl else "100_50_5"

print("==================================================")
print(f"🎯 Cấu Hình Thực Nghiệm Đã Chọn:")
print(f"• Model:             {MODEL_CHOICE.upper()} ({model_name})")
print(f"• Method:            {METHOD.upper()}")
print(f"• Particles (N):     {NUM_PARTICLES}")
print(f"• Guidance Scale:    {guidance_scale} (⚡ Auto-Dispatch chuẩn bài báo)")
if "rs_" in METHOD:
    print(f"• Smoothing Sigma:   {effective_sigma}")
    print(f"• Monte Carlo M:     {effective_m}")
    print(f"• CRN Invariance:    {SMC_USE_CRN}")
print(f"• Resample Interval: Mỗi {RESAMPLE_FREQUENCY} bước (i ∈ [20, 80])")
print(f"• SMC Lambda:        {FK_LMBDA}")
print(f"• Save to Drive:     {SAVE_TO_DRIVE and DRIVE_SMC_DIR is not None}")
print("==================================================")


## 6. 🚀 Thực Thi Chạy Đơn Lẻ (Single Run Executor)


In [ ]:
import os, shutil

if not os.path.exists("LiDAR_sampling.py"):
    %cd /content/RS-LiDAR

# Thiết lập danh sách cờ dòng lệnh
cmd = [
    "python", "LiDAR_sampling.py",
    "--model_name", f"'{model_name}'",
    "--scale", str(guidance_scale),
    "--num_particles", str(NUM_PARTICLES),
    "--num_inference_steps", str(NUM_INFERENCE_STEPS),
    "--resample_frequency", str(RESAMPLE_FREQUENCY),
    "--FK_lmbda", str(FK_LMBDA),
]

# Thêm cờ prompt
if PROMPT_SELECTION_MODE == "stratified_by_category":
    import json
    meta_path = "prompt_files/geneval_metadata.jsonl"
    with open(meta_path, "r", encoding="utf-8") as f:
        prompts = [json.loads(line) for line in f if line.strip()]
    cats = ["single_object", "two_object", "counting", "colors", "position", "color_attr"]
    selected_indices = []
    for cat in cats:
        matched = [i for i, p in enumerate(prompts) if p.get("category") == cat]
        selected_indices.extend(matched[:PROMPTS_PER_CATEGORY])
    cmd.extend(["--prompt_indices", f"'{','.join(map(str, selected_indices))}'"])
elif PROMPT_SELECTION_MODE == "custom_indices":
    cmd.extend(["--prompt_indices", f"'{CUSTOM_PROMPT_INDICES}'"])
else:
    cmd.extend(["--max_prompt", str(NUM_PROMPTS)])

# Cấu hình cờ phương pháp
if METHOD == "vanilla_smc":
    cmd.append("--use_smc")
elif METHOD == "rs_smc":
    cmd.extend([
        "--use_rs_smc",
        "--smc_sigma", str(effective_sigma),
        "--smc_num_mc_samples", str(effective_m),
    ])
    if SMC_USE_CRN:
        cmd.append("--smc_use_crn")
elif METHOD == "bon":
    pass # Best-of-N tự động được trích xuất từ metrics_arr['ImageReward']['max']
elif METHOD == "smc_lidar":
    cmd.extend([
        "--use_smc",
        "--use_rag",
        "--lookahead_path", lookahead_dataset,
    ])
elif METHOD == "rs_smc_lidar":
    cmd.extend([
        "--use_rs_smc",
        "--smc_sigma", str(effective_sigma),
        "--smc_num_mc_samples", str(effective_m),
        "--use_rag",
        "--lookahead_path", lookahead_dataset,
    ])
    if SMC_USE_CRN:
        cmd.append("--smc_use_crn")

if RUN_NAME.strip():
    cmd.extend(["--run_name", f"'{RUN_NAME.strip()}'"])

print(f"🚀 Đang chạy lệnh:\n{' '.join(cmd)}\n")
!{' '.join(cmd)}

# Tự động đồng bộ thư mục kết quả mới nhất sang Google Drive
if SAVE_TO_DRIVE and DRIVE_SMC_DIR is not None and os.path.exists("Target_samples"):
    subdirs = [os.path.join("Target_samples", d) for d in os.listdir("Target_samples") if os.path.isdir(os.path.join("Target_samples", d))]
    if subdirs:
        latest_subdir = max(subdirs, key=os.path.getmtime)
        dest_dir = os.path.join(DRIVE_SMC_DIR, os.path.basename(latest_subdir))
        shutil.copytree(latest_subdir, dest_dir, dirs_exist_ok=True)
        print(f"\n☁️ Đã đồng bộ an toàn thư mục kết quả về Google Drive:\n📁 {dest_dir}")


## 7. 🔄 Chạy Thử Nghiệm Sweep Bảng 8 (Table 8 Comparative Sweep Runner)
> Cell này sẽ tự động chạy so sánh trực diện giữa **Vanilla SMC** và **RS-SMC** trên các mức hạt $N \in \{2, 4, 8, 16\}$, tự động trích xuất điểm ImageReward trung bình (`mean` cho SMC) và cực đại (`max` cho SMC + BoN).


In [ ]:
# @title ⚡ Auto Table 8 Comparative Sweep (Vanilla SMC vs RS-SMC)
RUN_SWEEP = False # @param {type:"boolean"}
SWEEP_PARTICLES = [2, 4, 8] # @param {type:"raw"}
# @markdown *(Gợi ý: [2, 4, 8] để chạy nhanh trong 15-20 phút; Thêm 16 nếu có GPU A100)*

if RUN_SWEEP:
    if not os.path.exists("LiDAR_sampling.py"):
        %cd /content/RS-LiDAR

    sweep_results = {"Vanilla_SMC": {}, "RS_SMC": {}}
    prompt_arg = f"--prompt_indices '{CUSTOM_PROMPT_INDICES}'" if PROMPT_SELECTION_MODE == "custom_indices" else f"--max_prompt {NUM_PROMPTS}"

    for n_p in SWEEP_PARTICLES:
        print(f"\n{'='*70}")
        print(f"🚀 BẮT ĐẦU CHẠY THỰC NGHIỆM VỚI SỐ HẠT N = {n_p}")
        print(f"{'='*70}\n")

        # 1. Chạy Vanilla SMC
        vanilla_run_name = f"{MODEL_CHOICE.upper()}_Sweep_Vanilla_SMC_N{n_p}"
        cmd_vanilla = f"python LiDAR_sampling.py --model_name '{model_name}' --scale {guidance_scale} --use_smc --num_particles {n_p} --num_inference_steps {NUM_INFERENCE_STEPS} --resample_frequency {RESAMPLE_FREQUENCY} --FK_lmbda {FK_LMBDA} {prompt_arg} --run_name '{vanilla_run_name}'"
        print(f"▶️ Chạy Vanilla SMC (N={n_p})...")
        !{cmd_vanilla}

        # 2. Chạy RS-SMC
        crn_flag = "--smc_use_crn" if SMC_USE_CRN else ""
        rs_run_name = f"{MODEL_CHOICE.upper()}_Sweep_RS_SMC_N{n_p}_M{effective_m}_sigma{effective_sigma}"
        cmd_rs = f"python LiDAR_sampling.py --model_name '{model_name}' --scale {guidance_scale} --use_rs_smc --smc_sigma {effective_sigma} --smc_num_mc_samples {effective_m} {crn_flag} --num_particles {n_p} --num_inference_steps {NUM_INFERENCE_STEPS} --resample_frequency {RESAMPLE_FREQUENCY} --FK_lmbda {FK_LMBDA} {prompt_arg} --run_name '{rs_run_name}'"
        print(f"▶️ Chạy RS-SMC (N={n_p}, M={effective_m}, sigma={effective_sigma})...")
        !{cmd_rs}

        # Đồng bộ về Drive nếu có
        if SAVE_TO_DRIVE and DRIVE_SMC_DIR is not None:
            for rname in [vanilla_run_name, rs_run_name]:
                src = os.path.join("Target_samples", rname)
                if os.path.exists(src):
                    shutil.copytree(src, os.path.join(DRIVE_SMC_DIR, rname), dirs_exist_ok=True)
            print(f"☁️ Đã đồng bộ kết quả N={n_p} về Google Drive.")
else:
    print("ℹ️ Đặt RUN_SWEEP = True và chạy lại Cell này để thực hiện sweep so sánh tự động!")


## 8. 📊 Trực Quan Hóa & Bảng Tổng Hợp Chuẩn Bảng 8 Bài Báo (Table 8 Visualization)


In [ ]:
# @title 📊 Hiển Thị Bảng So Sánh & Đồ Thị ImageReward Chuẩn Table 8
import os, glob, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate
from IPython.display import display, HTML

# Điểm tham chiếu bài báo gốc ICML 2026 (Table 8 trên SD v1.5)
PAPER_TABLE8 = {
    "SMC":                 {1: -0.001, 2: 0.001, 4: 0.267, 8: 0.568, 16: 0.864, 32: 1.033, 64: 1.129},
    "SMC + LiDAR":         {1: 0.384,  2: 0.715, 4: 1.007, 8: 1.122, 16: 1.234, 32: 1.307, 64: 1.338},
    "BoN":                 {1: -0.001, 2: 0.414, 4: 0.739, 8: 0.948, 16: 1.133, 32: 1.256, 64: 1.346},
    "BoN + LiDAR":         {1: 0.384,  2: 0.749, 4: 0.969, 8: 1.136, 16: 1.277, 32: 1.351, 64: 1.418},
    "SMC (+ BoN)":         {1: -0.001, 2: 0.336, 4: 0.692, 8: 1.015, 16: 1.201, 32: 1.331, 64: 1.438},
    "SMC (+ BoN) + LiDAR": {1: 0.384,  2: 0.770, 4: 1.073, 8: 1.218, 16: 1.354, 32: 1.452, 64: 1.510},
}

# Quét tất cả file final_metrics.json trong Target_samples
metric_files = glob.glob("Target_samples/**/final_metrics.json", recursive=True)

if metric_files:
    data_records = []
    for mf in metric_files:
        try:
            folder_name = os.path.basename(os.path.dirname(mf))
            with open(mf, "r", encoding="utf-8") as f:
                content = json.load(f)
            ir = content.get("ImageReward", {})
            mean_ir = ir.get("mean", None)
            max_ir = ir.get("max", None)
            
            # Đọc args.json nếu có
            args_f = os.path.join(os.path.dirname(mf), "args.json")
            n_part = None
            if os.path.exists(args_f):
                with open(args_f, "r", encoding="utf-8") as af:
                    a_dict = json.load(af)
                n_part = a_dict.get("num_particles")
                is_rs = a_dict.get("use_smc_smoothing", False)
                is_smc = a_dict.get("use_smc", False)
                is_rag = a_dict.get("use_rag", False)
                model_used = a_dict.get("model_name", "")
            else:
                is_rs = "RSSMC" in folder_name
                is_smc = "SMC" in folder_name
                is_rag = "LiDAR" in folder_name
                n_part = int(folder_name.split("_N")[1].split("_")[0]) if "_N" in folder_name else None
                model_used = "SDXL" if "SDXL" in folder_name else "SD15"
                
            if is_rs and is_rag:
                meth_name = "RS-SMC + LiDAR"
            elif is_rs:
                meth_name = "RS-SMC"
            elif is_smc and is_rag:
                meth_name = "SMC + LiDAR"
            elif is_smc:
                meth_name = "Vanilla SMC"
            elif is_rag:
                meth_name = "LiDAR"
            else:
                meth_name = "BoN" if (n_part and n_part > 1) else "Vanilla"

            data_records.append({
                "Folder": folder_name,
                "Model": "SDXL" if "xl" in model_used.lower() else "SD 1.5",
                "Method": meth_name,
                "Particles (N)": n_part,
                "SMC ImageReward (Mean)": f"{mean_ir:.4f}" if mean_ir is not None else "N/A",
                "BoN ImageReward (Max)": f"{max_ir:.4f}" if max_ir is not None else "N/A",
            })
        except Exception as e:
            continue

    if data_records:
        df = pd.DataFrame(data_records).sort_values(by=["Model", "Particles (N)", "Method"])
        print("\n" + "="*85)
        print("📊 BẢNG TỔNG HỢP KẾT QUẢ THỰC NGHIỆM ĐÃ ĐO (TABLE 8 FORMAT)")
        print("="*85)
        print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))
        print("="*85 + "\n")
        
        display(HTML("<h3>📊 Bảng Tổng Hợp Kết Quả Đo Đạc:</h3>"))
        display(HTML(df.to_html(escape=False, index=False)))

    # Vẽ đồ thị so sánh nếu có đủ dữ liệu
    df_valid = pd.DataFrame(data_records).dropna()
    if len(df_valid) >= 2 and any(df_valid["Method"] == "RS-SMC"):
        plt.figure(figsize=(9, 5))
        for m in df_valid["Method"].unique():
            sub = df_valid[df_valid["Method"] == m].sort_values(by="Particles (N)")
            x_vals = sub["Particles (N)"].astype(int)
            y_vals = sub["SMC ImageReward (Mean)"].astype(float)
            plt.plot(x_vals, y_vals, marker="o", linewidth=2.5, label=f"{m} (Mean IR)")
        
        plt.title("Scaling ImageReward with Particles (N): Vanilla SMC vs RS-SMC", fontsize=13, fontweight="bold")
        plt.xlabel("Number of Particles (N)", fontsize=11)
        plt.ylabel("ImageReward", fontsize=11)
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.legend(fontsize=11)
        plt.tight_layout()
        plt.show()
else:
    print("ℹ️ Chưa tìm thấy file 'final_metrics.json' nào trong 'Target_samples/'. Vui lòng chạy Cell 6 hoặc Cell 7 để đo đạc kết quả!")
